In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tuning import Tuner
from strategy import PairsTradingStrategy


In [ ]:
# Clean Data

PAIRS = [
    ("V", "MA", "Data/V_daily.csv", "Data/MA_daily.csv"),
    ("PEP", "KO", "Data/PEP_daily.csv", "Data/KO_daily.csv"),
    ("XOM", "CVX", "Data/XOM_daily.csv", "Data/CVX_daily.csv"),
    ("JPM", "BAC", "Data/JPM_daily.csv", "Data/BAC_daily.csv"),
    ("GOOG", "GOOGL", "Data/GOOG_daily.csv", "Data/GOOGL_daily.csv"),
    ("SPY", "IVV", "Data/SPY_daily.csv", "Data/IVV_daily.csv"),
    ("GLD", "IAU", "Data/GLD_daily.csv", "Data/IAU_daily.csv"),
    ("AGG", "BND", "Data/AGG_daily.csv", "Data/BND_daily.csv"),
    ("VTI", "SCHB", "Data/VTI_daily.csv", "Data/VTI_daily.csv"),
]


def load_yfinance_csv(path, ticker):
    asset = pd.read_csv(path)
    asset = asset.iloc[2:].copy()
    asset = asset.rename(columns={"Price": "Date", "Adj Close": ticker})
    asset = asset[["Date", ticker]].copy()
    asset["Date"] = pd.to_datetime(asset["Date"], errors="coerce")
    asset[ticker] = pd.to_numeric(asset[ticker], errors="coerce")
    asset = asset.dropna(subset=["Date", ticker])
    return asset


pair_data = {}

for y_ticker, x_ticker, y_path, x_path in PAIRS:
    y_asset = load_yfinance_csv(y_path, y_ticker)
    x_asset = load_yfinance_csv(x_path, x_ticker)

    pair_df = y_asset.merge(x_asset, on="Date", how="inner")
    pair_df = pair_df.sort_values("Date").reset_index(drop=True)
    pair_df = pair_df[(pair_df[y_ticker] > 0) & (pair_df[x_ticker] > 0)].copy()

    pair_df[f"log_{y_ticker}"] = np.log(pair_df[y_ticker])
    pair_df[f"log_{x_ticker}"] = np.log(pair_df[x_ticker])
    pair_df = pair_df.dropna().reset_index(drop=True)

    pair_data[(y_ticker, x_ticker)] = pair_df

pair_data

In [ ]:
# Backtest PairsTradingStrategy from 2023-01-01 through 2025-12-31

BACKTEST_START = pd.Timestamp("2023-01-01")
BACKTEST_END = pd.Timestamp("2025-12-31")
TRAILING_TUNE_DAYS = 756  # roughly 3 calendar years
MAX_ACTIVE_PAIRS = 5
MIN_TUNE_SHARPE = 0.0
MIN_TUNE_ENTRIES = 5

TRADING_DAYS_PER_YEAR = 252
RISK_FREE_RATE_ANNUAL = 0.04
CAPITAL_BASE = 1.0

trading_dates = sorted(
    set().union(*[
        set(df.loc[(df["Date"] >= BACKTEST_START) & (df["Date"] <= BACKTEST_END), "Date"])
        for df in pair_data.values()
    ])
)

month_starts = pd.Series(trading_dates).groupby(pd.Series(trading_dates).dt.to_period("M")).min().tolist()

monthly_tuning_history = []
daily_decisions = []
strategy = None

for current_date in trading_dates:
    current_date = pd.Timestamp(current_date)

    if current_date in month_starts or strategy is None:
        tune_end_date = current_date - pd.Timedelta(days=1)
        monthly = Tuner.tune_universe(
            pair_data=pair_data,
            pairs=list(pair_data.keys()),
            end_date=tune_end_date,
            trailing_window_days=TRAILING_TUNE_DAYS,
            min_sharpe=MIN_TUNE_SHARPE,
            min_entries=MIN_TUNE_ENTRIES,
            max_pairs=MAX_ACTIVE_PAIRS,
        )

        monthly_tuning_history.append({
            "rebalance_date": current_date,
            "tune_start_date": tune_end_date - pd.DateOffset(days=TRAILING_TUNE_DAYS),
            "tune_end_date": tune_end_date,
            "active_pairs": monthly["active_pairs"],
            "num_active_pairs": len(monthly["active_pairs"]),
            "failures": monthly["failures"],
            "tuned_params": monthly["tuned_params"],
            "all_tuned_params": monthly.get("all_tuned_params", pd.DataFrame()),
        })

        if strategy is None:
            strategy = PairsTradingStrategy(
                pair_data=pair_data,
                tuned_params=monthly["tuned_params"],
                active_pairs=monthly["active_pairs"],
                regime_lookback=252,
                target_spread_vol=0.01,
                max_pair_position=1.0,
                max_gross_exposure=4.0,
                transaction_cost_bps=1.0,
            )
        else:
            strategy.update_tuned_params(monthly["tuned_params"])
            strategy.set_active_pairs(monthly["active_pairs"])

        print(f"{current_date.date()} active pairs: {monthly['active_pairs']}")

    if strategy is None or len(strategy.active_pairs) == 0:
        continue

    day_decisions = strategy.run_day(current_date)
    if not day_decisions.empty:
        daily_decisions.append(day_decisions)

if daily_decisions:
    backtest_trades = pd.concat(daily_decisions, ignore_index=True)
else:
    backtest_trades = pd.DataFrame()

monthly_tuning_summary = pd.DataFrame([
    {
        "rebalance_date": row["rebalance_date"],
        "tune_start_date": row["tune_start_date"],
        "tune_end_date": row["tune_end_date"],
        "num_active_pairs": row["num_active_pairs"],
        "active_pairs": row["active_pairs"],
        "num_failures": len(row["failures"]),
    }
    for row in monthly_tuning_history
])

display(monthly_tuning_summary)
display(backtest_trades.head())


In [ ]:
# Compute daily portfolio P&L, excess returns, and performance metrics

if backtest_trades.empty:
    raise ValueError("No trades/decisions were generated. Check tuning filters and data coverage.")

backtest_trades = backtest_trades.sort_values(["pair", "date"]).reset_index(drop=True)
backtest_trades["spread_change"] = backtest_trades.groupby("pair")["spread"].diff()
backtest_trades["gross_pnl"] = backtest_trades["previous_position"] * backtest_trades["spread_change"].fillna(0)
backtest_trades["net_pnl"] = backtest_trades["gross_pnl"] - backtest_trades["transaction_cost"]

portfolio_daily = (
    backtest_trades
    .groupby("date", as_index=False)
    .agg(
        gross_pnl=("gross_pnl", "sum"),
        transaction_cost=("transaction_cost", "sum"),
        net_pnl=("net_pnl", "sum"),
        gross_exposure=("target_position", lambda x: x.abs().sum()),
        active_positions=("target_position", lambda x: x.ne(0).sum()),
        regime_pass_rate=("regime_passed", "mean"),
    )
)
portfolio_daily["gross_return"] = portfolio_daily["gross_pnl"] / CAPITAL_BASE
portfolio_daily["net_return"] = portfolio_daily["net_pnl"] / CAPITAL_BASE
portfolio_daily["risk_free_daily"] = (1 + RISK_FREE_RATE_ANNUAL) ** (1 / TRADING_DAYS_PER_YEAR) - 1
portfolio_daily["gross_excess_return"] = portfolio_daily["gross_return"] - portfolio_daily["risk_free_daily"]
portfolio_daily["net_excess_return"] = portfolio_daily["net_return"] - portfolio_daily["risk_free_daily"]

portfolio_daily["cum_gross_pnl"] = portfolio_daily["gross_pnl"].cumsum()
portfolio_daily["cum_net_pnl"] = portfolio_daily["net_pnl"].cumsum()
portfolio_daily["drawdown"] = portfolio_daily["cum_net_pnl"] - portfolio_daily["cum_net_pnl"].cummax()

gross_vol = portfolio_daily["gross_return"].std()
net_vol = portfolio_daily["net_return"].std()
gross_excess_sharpe = np.nan if gross_vol == 0 or pd.isna(gross_vol) else np.sqrt(TRADING_DAYS_PER_YEAR) * portfolio_daily["gross_excess_return"].mean() / gross_vol
net_excess_sharpe = np.nan if net_vol == 0 or pd.isna(net_vol) else np.sqrt(TRADING_DAYS_PER_YEAR) * portfolio_daily["net_excess_return"].mean() / net_vol

performance_summary = pd.DataFrame([{
    "start_date": portfolio_daily["date"].min(),
    "end_date": portfolio_daily["date"].max(),
    "trading_days": len(portfolio_daily),
    "capital_base": CAPITAL_BASE,
    "risk_free_rate_annual": RISK_FREE_RATE_ANNUAL,
    "risk_free_rate_daily": portfolio_daily["risk_free_daily"].iloc[0],
    "total_gross_pnl": portfolio_daily["gross_pnl"].sum(),
    "total_transaction_cost": portfolio_daily["transaction_cost"].sum(),
    "total_net_pnl": portfolio_daily["net_pnl"].sum(),
    "annual_net_return": portfolio_daily["net_return"].mean() * TRADING_DAYS_PER_YEAR,
    "annual_net_excess_return": portfolio_daily["net_excess_return"].mean() * TRADING_DAYS_PER_YEAR,
    "daily_gross_vol": gross_vol,
    "daily_net_vol": net_vol,
    "gross_excess_sharpe": gross_excess_sharpe,
    "net_excess_sharpe": net_excess_sharpe,
    "max_drawdown": portfolio_daily["drawdown"].min(),
    "avg_gross_exposure": portfolio_daily["gross_exposure"].mean(),
    "avg_active_positions": portfolio_daily["active_positions"].mean(),
    "avg_regime_pass_rate": portfolio_daily["regime_pass_rate"].mean(),
}])

strategy_sharpe = pd.DataFrame([{
    "risk_free_rate_annual": RISK_FREE_RATE_ANNUAL,
    "gross_excess_sharpe": gross_excess_sharpe,
    "net_excess_sharpe": net_excess_sharpe,
}])

print(f"Risk-free rate assumption: {RISK_FREE_RATE_ANNUAL:.2%} annual")
print(f"Strategy gross excess Sharpe: {gross_excess_sharpe:.4f}")
print(f"Strategy net excess Sharpe: {net_excess_sharpe:.4f}")
display(strategy_sharpe.round(6))
display(performance_summary.round(6))
display(portfolio_daily.tail())


In [ ]:
# Plot cumulative P&L, drawdown, and exposure

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(portfolio_daily["date"], portfolio_daily["cum_gross_pnl"], label="Gross P&L")
axes[0].plot(portfolio_daily["date"], portfolio_daily["cum_net_pnl"], label="Net P&L")
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Pairs Strategy Cumulative P&L")
axes[0].set_ylabel("Spread P&L")
axes[0].legend()

axes[1].fill_between(portfolio_daily["date"], portfolio_daily["drawdown"], 0, color="tab:red", alpha=0.35)
axes[1].set_title("Net P&L Drawdown")
axes[1].set_ylabel("Drawdown")

axes[2].plot(portfolio_daily["date"], portfolio_daily["gross_exposure"], label="Gross exposure")
axes[2].plot(portfolio_daily["date"], portfolio_daily["active_positions"], label="Active positions")
axes[2].set_title("Exposure")
axes[2].set_ylabel("Count / Gross")
axes[2].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Pair-level diagnostics

pair_summary = (
    backtest_trades
    .groupby("pair", as_index=False)
    .agg(
        total_gross_pnl=("gross_pnl", "sum"),
        total_transaction_cost=("transaction_cost", "sum"),
        total_net_pnl=("net_pnl", "sum"),
        avg_abs_position=("target_position", lambda x: x.abs().mean()),
        active_days=("target_position", lambda x: x.ne(0).sum()),
        regime_pass_rate=("regime_passed", "mean"),
        avg_adf_pvalue=("adf_pvalue", "mean"),
        avg_half_life=("half_life", "mean"),
    )
    .sort_values("total_net_pnl", ascending=False)
    .reset_index(drop=True)
)

display(pair_summary.round(6))

active_pair_history = monthly_tuning_summary[["rebalance_date", "num_active_pairs", "active_pairs"]]
print(pair_summary.round(6))
print(active_pair_history)